# Prompt Engineering Certification — Hands-On Lab Notebook
## All 6 Modules | Sessions 1–30
### UpSkill Global Education Technologies Inc., Canada

---

| Module | Sessions | Topics |
|--------|----------|--------|
| 1 | 1–5 | AI Basics, Tokens, Prompt Anatomy, Best Practices, CRAFT |
| 2 | 6–10 | Zero-Shot, Few-Shot, Chain-of-Thought, Persona, Optimization |
| 3 | 11–15 | Research, Writing, Presentations, Email, Productivity |
| 4 | 16–20 | Marketing, HR, Finance, Strategy, Customer Support |
| 5 | 21–25 | Workflows, Frameworks, Automation, Prompt Library, Responsible AI |
| 6 | 26–30 | Capstone Build, QA, Presentation Prep, Certification |

**Prerequisites:** `pip install openai python-dotenv pandas`

In [ ]:
# GLOBAL SETUP — Run before any lab
import os, json, time, datetime
from IPython.display import Markdown, display
from openai import OpenAI
API_KEY = "sk-your-api-key-here"  # Replace with your key
client = OpenAI(api_key=API_KEY)
def ask(prompt, model='gpt-4o-mini', temperature=0.7, max_tokens=800, system=None):
    messages = []
    if system: messages.append({'role':'system','content':system})
    messages.append({'role':'user','content':prompt})
    r = client.chat.completions.create(model=model,messages=messages,temperature=temperature,max_tokens=max_tokens)
    return r.choices[0].message.content
def show(text, title=None):
    if title: display(Markdown(f'### {title}'))
    display(Markdown(text))
print('Setup complete.')

---
# MODULE 1 — Foundations of Prompt Engineering
## Sessions 1–5

## Session 1 — Temperature Effect
**Task:** Same prompt at 0.0 vs 1.0 — compare outputs.

In [ ]:
prompt = 'Explain Artificial Intelligence in 3 sentences for a non-technical business executive.'
print('=== TEMPERATURE 0.0 ===')
print(ask(prompt, temperature=0.0))
print()
print('=== TEMPERATURE 1.0 ===')
print(ask(prompt, temperature=1.0))
print('REFLECTION: Which suits a board presentation? Run 3 times — which varies more?')

In [ ]:
# AI Hierarchy via analogy
p = ('Explain the relationship between AI, Machine Learning, Deep Learning, and LLMs '
     'using a single analogy. Clear for a non-technical professional. Exactly 4 bullet points.')
show(ask(p, temperature=0.5), 'AI Hierarchy via Analogy')

## Session 2 — Tokens & Context Windows

In [ ]:
r = client.chat.completions.create(model='gpt-4o-mini',
    messages=[{'role':'user','content':'Explain tokenization in LLMs simply.'}],
    temperature=0.5, max_tokens=300)
u = r.usage
print(r.choices[0].message.content)
print(f'\nPrompt: {u.prompt_tokens} | Completion: {u.completion_tokens} | Total: {u.total_tokens}')
cost = (u.prompt_tokens*0.00015 + u.completion_tokens*0.0006)/1000
print(f'Est. cost: ${cost:.6f}')

In [ ]:
# Multi-turn context window demo
conv = [{'role':'system','content':'Concise AI tutor. Keep replies under 80 words.'},
        {'role':'user','content':'What is a token in LLMs?'}]
r1 = client.chat.completions.create(model='gpt-4o-mini',messages=conv,max_tokens=150)
reply1 = r1.choices[0].message.content
print('Turn 1:', reply1)
conv.append({'role':'assistant','content':reply1})
conv.append({'role':'user','content':'Give one real-world example of token limits causing a problem.'})
r2 = client.chat.completions.create(model='gpt-4o-mini',messages=conv,max_tokens=150)
print('Turn 2:', r2.choices[0].message.content)
print(f'Total tokens: {r1.usage.total_tokens + r2.usage.total_tokens}')

## Session 3 — Prompt Anatomy (ROLE | CONTEXT | TASK | FORMAT | CONSTRAINTS)

In [ ]:
ROLE='You are a senior business analyst with 15 years of retail strategy experience.'
CONTEXT='A mid-size Indian retail chain (500Cr revenue, 80 stores) is considering e-commerce entry.'
TASK='Write a concise risk assessment of this e-commerce entry decision.'
FORMAT='3 sections: Financial Risks, Operational Risks, Competitive Risks — 2 bullets each.'
CONSTRAINTS='Under 250 words. No jargon. Specific to this company.'
full = f'{ROLE}\n\nCONTEXT: {CONTEXT}\n\nTASK: {TASK}\n\nFORMAT: {FORMAT}\n\nCONSTRAINTS: {CONSTRAINTS}'
show(ask(full, temperature=0.4), '5-Component Prompt Output')

In [ ]:
weak = 'Tell me about digital transformation.'
strong = ('You are a digital transformation consultant advising the CTO of a 200-person manufacturing company.\n'
          'Define digital transformation in 2 sentences relevant to manufacturing.\n'
          'List the 3 highest-priority starting points with one specific action each. Under 200 words.')
print('WEAK:'); print(ask(weak, temperature=0.5))
print('\n'+'='*60+'\n')
print('STRONG:'); show(ask(strong, temperature=0.5))

## Session 4 — Prompt Best Practices

In [ ]:
p_basic = 'Write about the benefits of remote work.'
p_best = ('Evaluate the top 4 benefits of remote work for knowledge workers in tech companies.\n'
          'For each: bold heading + one specific measurable example + one manager implication.\n'
          'Do NOT use generic statements without evidence. Under 300 words.')
show(ask(p_basic), 'Basic Output')
print('\n'+'='*60+'\n')
show(ask(p_best, temperature=0.4), 'Best-Practices Output')

In [ ]:
# Prompt Decomposition: complex task into 2 steps
notes = ('New product: AI inventory forecasting for SME retailers.\n'
         'Launch Q1. Target: 500 pilot customers. Price: 4999/month.\n'
         'Differentiator: Works with existing POS, no IT team needed.\n'
         'Risk: Competitor has 18-month head start.')
kp = ask(f'Extract 5 most important strategic points as a numbered list. One sentence each.\n\n{notes}', temperature=0.2)
print('STEP 1 — Key Points:'); print(kp)
es = ask(f'Write a 150-word executive summary for a Board audience. 2 paragraphs. Professional tone.\n\nPoints:\n{kp}', temperature=0.4)
print('\nSTEP 2 — Executive Summary:'); show(es)

## Session 5 — CRAFT Framework (Context | Role | Action | Format | Tone)

In [ ]:
def craft(context, role, action, fmt, tone):
    return f'ROLE: {role}\n\nCONTEXT: {context}\n\nACTION: {action}\n\nFORMAT: {fmt}\n\nTONE: {tone}'
p_a = craft(
    context='Company introducing WFH 3 days/week from next month.',
    role='You are the Chief People Officer writing an internal announcement.',
    action='Write the company-wide announcement for the new flexible work policy.',
    fmt='Email format. Subject + 3 paragraphs + closing. Max 200 words.',
    tone='Warm, inclusive, transparent — acknowledge both excitement and concerns.')
show(ask(p_a, temperature=0.6), 'CRAFT Output A — HR Announcement')
print('\n'+'='*60+'\n')
p_b = craft(
    context='SaaS platform had 4-hour outage affecting 2,000 customers. Cause: failed DB migration.',
    role='You are VP of Customer Success writing a public apology.',
    action='Write the customer apology email.',
    fmt='Email. Subject + 3 paragraphs: apology, explanation, remedy. Max 180 words.',
    tone='Accountable, sincere, action-oriented — no corporate deflection.')
show(ask(p_b, temperature=0.5), 'CRAFT Output B — Apology')

In [ ]:
# CRAFT audit: score and improve a weak prompt
weak = 'Write a marketing email for our new product.'
audit = ('Evaluate this prompt against CRAFT (C=Context,R=Role,A=Action,F=Format,T=Tone).\n'
         'Score each 1-5 with a one-sentence reason. Output as markdown table.\n'
         f'Then write an improved version scoring 5/5.\n\nPrompt: "{weak}"')
show(ask(audit, temperature=0.3), 'CRAFT Audit + Improved Prompt')

---
# MODULE 2 — Prompting Techniques
## Sessions 6–10

## Session 6 — Zero-Shot Prompting

In [ ]:
reviews = ['The delivery was late but product quality exceeded my expectations.',
           'Absolutely terrible experience. Never ordering again.',
           'It was okay. Nothing special but did the job.']
print('ZERO-SHOT SENTIMENT CLASSIFICATION')
for r in reviews:
    res = ask(f'Classify as Positive, Negative, or Mixed. ONLY label + 5-word reason.\nReview: "{r}"', temperature=0.0)
    print(f'  {r[:55]}...\n  => {res}\n')
contract = ('Agreement 15 March 2024 between TechFlow Pvt Ltd and Nexus Retail Ltd. '
            'Cloud services 24 months. Fee: 85000/month+GST. Payment: Net 30. Termination: 90 days.')
schema = '{"parties":[],"start_date":"","duration":"","monthly_fee":"","payment_terms":"","termination_notice":""}'
print('ZERO-SHOT EXTRACTION:')
print(ask(f'Extract as JSON: {schema}.\nContract: {contract}', temperature=0.0))

## Session 7 — Few-Shot Prompting

In [ ]:
few_shot = (
    'You write complaint responses: (1) Acknowledge specific problem (2) One apology (3) Exact resolution+timeline (4) Close with name.\n\n'
    'EXAMPLE 1:\nCOMPLAINT: Order #4521 arrived damaged.\n'
    'RESPONSE: Your order #4521 arrived damaged — entirely unacceptable. I have arranged a replacement within 48 hours. Thank you, Priya.\n\n'
    'EXAMPLE 2:\nCOMPLAINT: Charged twice for subscription. 2999 appeared twice.\n'
    'RESPONSE: You were double-charged 2999 — our error, I apologise. Full refund in 5 business days. Thank you, Rahul.\n\n'
    'Now respond to:\nCOMPLAINT: Waited 3 weeks for laptop repair. No updates. Need it for work.\nCUSTOMER: Anjali\nRESPONSE:')
show(ask(few_shot, temperature=0.4), 'Few-Shot Response')
print('CHECK: 1) Specific problem? 2) Single apology? 3) Resolution+timeline? 4) Name used?')

## Session 8 — Chain-of-Thought Prompting

In [ ]:
scenario = ('FMCG company considering premium organic snack launch.\n'
            'Revenue: 120Cr. Organic CAGR: 28%. Manufacturing cost +40%. Price: 80 vs 30.\n'
            'Marketing: 8Cr Year 1. Competitor: 14-month head start.')
print('DIRECT ANSWER:')
print(ask(scenario+'\nShould they launch? 2 sentences.', temperature=0.3))
cot = (scenario + '\n\nThink step by step:\nStep 1: Market opportunity\nStep 2: Financial viability\nStep 3: Competitive risk\nStep 4: Recommendation with one condition')
print('\nCHAIN-OF-THOUGHT:')
show(ask(cot, temperature=0.3))

In [ ]:
# Self-Consistency: run CoT 3 times
problem = ('Sales team: 8 people, 12 calls/day, 22 days/month.\n'
           'Conversion: 4% calls-to-meeting. Close rate: 25%. Avg deal: 150000.\n'
           'Think step by step: expected monthly revenue?')
for i in range(3):
    print(f'--- Run {i+1} ---'); print(ask(problem, temperature=0.3)); print()
print('REFLECTION: Do all 3 reach the same answer?')

## Session 9 — Persona Prompting

In [ ]:
q = 'What are the biggest risks of adopting AI in a mid-size enterprise?'
personas = {
    'CTO': 'You are a CTO with 20 years of enterprise tech leadership.',
    'CFO': 'You are a CFO focused on ROI, TCO, and financial risk.',
    'Employee Rep': 'You represent employees focused on workforce impact and job security.'
}
for role, persona in personas.items():
    print(f'\n{"="*50}\nPERSONA: {role}\n{"="*50}')
    print(ask(f'{persona}\n\nAnswer in 100 words:\n{q}', temperature=0.5))

In [ ]:
panel = ('Facilitate an expert panel. Three experts respond:\n'
         '1. DR. MEERA NAIR (AI Ethics): Responsible AI, bias, societal impact\n'
         '2. VIKRAM SHARMA (Founder): Practical application, speed, competitive edge\n'
         '3. PRIYA DESAI (Risk Manager): Governance, compliance, risk mitigation\n\n'
         'Question: Should Indian companies adopt GenAI immediately or wait for regulation?\n'
         'Format: Each expert 50 words. Then 30-word synthesis of agreement and difference.')
show(ask(panel, temperature=0.6), 'Expert Panel')

## Session 10 — Prompt Optimization

In [ ]:
article = ('India manufacturing PMI 56.4 (16-year high, Dec 2024). Autos and electronics led; capacity 82%.\n'
           'Input inflation +4.2% MoM. ICICI: margin pressure could slow Q1 2025 growth.')
v1 = f'Summarise this article.\n{article}'
v2 = ('Financial analyst briefing investment committee.\n'
      'Summarise in 3 bullets: What happened | Why it matters | Watch out for\n\n'
      f'Article: {article}')
v3 = v2 + '\nAfter writing, rate: Specificity (1-5) and Actionability (1-5). If either <4, rewrite that bullet.'
for label, p in [('v1.0 Basic',v1),('v2.0 Structured',v2),('v3.0 Self-Critique',v3)]:
    print(f'\n{"="*50}\n{label}\n{"="*50}'); print(ask(p, temperature=0.3))

---
# MODULE 3 — AI for Professional Applications
## Sessions 11–15

## Session 11 — Research & Knowledge Synthesis

In [ ]:
s1='Gartner 2024: 65% enterprises use GenAI by 2025. Content 72%, code 61%, CX 54%.'
s2='McKinsey 2024: 79% use AI. Only 28% have governance. Productivity gains 15-20%.'
s3='NASSCOM 2024: India AI investment +45% YoY. Talent shortage #1 (68% CIOs). Need 1.2M by 2026; have 420K.'
p = ('Research analyst briefing C-suite at Indian tech company.\n'
     'Synthesise 3 sources. Identify CROSS-SOURCE themes — do NOT summarise each separately.\n\n'
     f'A: {s1}\nB: {s2}\nC: {s3}\n\n'
     '1. CONSENSUS 2. KEY TENSION 3. INDIA IMPLICATION 4. WATCH AREA\nMax 250 words. Cite A/B/C.')
show(ask(p, temperature=0.4), 'Research Synthesis')

## Session 12 — AI for Professional Writing

In [ ]:
topic='Implementing a Hybrid Work Policy: Recommendations for 2025'
ctx='250-person tech company, 60% WFH-eligible, attrition 18%'
outline = ask(f'Senior HR consultant. 5-section report outline.\nTopic: {topic}\nContext: {ctx}\nSection title + 2-line description only.', temperature=0.3)
print('OUTLINE:'); print(outline)
es = ask(f'Senior HR consultant. 150-word executive summary.\nTopic: {topic}\nContext: {ctx}\nInclude: business case, attrition as driver, 3 recommendations preview, Q1 decision call.', temperature=0.4)
print('\nEXECUTIVE SUMMARY:'); show(es)
polished = ask(f'Edit: eliminate passive voice, replace jargon, strengthen opening.\n\n{es}', temperature=0.3)
print('\nPOLISHED:'); show(polished)

## Session 13 — AI for Presentations

In [ ]:
report = ('CSAT dropped 4.2 to 3.6 (Q2-Q3 2024). Response time 4 to 11hrs; FCR 78 to 61%.\n'
          'Tickets +30% post-August update. SMB churn +3.2pp. Fix: 4 agents + AI triage by Q4.')
p = ('Convert to 3 slides. For each: TITLE (insight headline) | BODY (3 bullets, max 10 words) | VISUAL (chart type)\n'
     'Slide 1: Problem | Slide 2: Root Cause | Slide 3: Actions\n\n'
     f'Report: {report}')
show(ask(p, temperature=0.4), 'Slide Content')

## Session 14 — AI for Email & Communication

In [ ]:
scenarios = [
    ('Deadline Extension', 'Asking manager for 1-week extension due to team illness', 'Professional, honest, solution-oriented'),
    ('Bad News to Client', 'Informing client of 2-week delay due to vendor issue', 'Accountable, empathetic, action-focused'),
    ('Cold Introduction',  'Reaching out to VP Sales at a potential partner firm',  'Confident, concise, value-first'),
]
for etype, ctx, tone in scenarios:
    p = (f'Professional communication coach.\nEmail for: {etype}\nContext: {ctx}\nTone: {tone}\n'
         'Format: Subject + body (max 150 words). Use [PLACEHOLDER] for names/dates.')
    print(f'\n{"="*50}\n{etype}\n{"="*50}'); print(ask(p, temperature=0.5))

## Session 15 — AI Productivity System

In [ ]:
tasks = ('1. Weekly status report (2 hrs)\n2. ~40 emails (3 hrs)\n3. 5 meeting agendas (1.5 hrs)\n'
         '4. Dashboard from 3 systems (2.5 hrs)\n5. Performance feedback (1 hr)\n'
         '6. Competitor research (2 hrs)\n7. LinkedIn 2 posts (1.5 hrs)\n8. Monthly review slides (3 hrs)')
p = ('AI productivity consultant auditing my weekly tasks.\n'
     'For each: AI Potential (HIGH/MEDIUM/LOW) | Prompt type | Est. time saving\n'
     'Output as markdown table. Then Top 3 quick wins with one action each.\n\n'
     f'Tasks:\n{tasks}')
show(ask(p, temperature=0.3), 'AI Productivity Audit')

---
# MODULE 4 — AI for Business & Industry Applications
## Sessions 16–20

## Session 16 — Marketing & Content Creation

In [ ]:
blog = ('Why 80% of Customer Churn Is Predictable.\n'
        'Three signals appear 60-90 days before churn: login <2x/week, tickets shift to billing, NPS <7.\n'
        'Personal outreach retains 3x more at-risk customers.\n'
        'A 15-min CSM call in first 30 days of decline recovers 68% of at-risk accounts.')
p = ('Content strategist. Brand voice: direct, evidence-based.\n'
     f'Source blog: {blog}\n\n'
     '1. LINKEDIN POST (180-220 words): hook + 3 insights + engagement question\n'
     '2. TWITTER THREAD (6 tweets, max 240 chars each): hook → 4 insights → CTA\n'
     '3. EMAIL SUBJECT LINES (3): curiosity-gap / data-led / benefit-first')
show(ask(p, temperature=0.6, max_tokens=900), 'Repurposed Content')

## Session 17 — AI for HR

In [ ]:
notes = ('Role: Data Analyst, Marketing Analytics, mid-level.\n'
         'Must have: SQL, Python, Excel, presenting to non-technical teams.\n'
         'Nice to have: Tableau/Power BI. Responsibilities: dashboards, campaign analysis, A/B testing.')
jd = ask(f'Senior HRBP. Write an inclusive JD. Lead with impact. Separate Must/Nice-to-Have. No unnecessary degrees.\nFormat: Title|About|Responsibilities(5)|Requirements|Benefits\nNotes: {notes}', temperature=0.4)
print('JD:'); print(jd)
bias = ask(f'Audit this JD for bias: masculine-coded language, ageist terms, over-qualification, cultural exclusion.\nFor each: |Issue|Type|Fix|. If none: No bias detected.\nJD: {jd}', temperature=0.1)
print('\nBIAS AUDIT:'); show(bias)

## Session 18 — AI for Finance & Data

In [ ]:
code_p = ('FP&A analyst who writes clean Python. Write a function taking DataFrame with [month, actual_revenue, plan_revenue].\n'
          'Add: variance, variance_pct, status (Favorable/Unfavorable/On Plan). Print summary totals.\n'
          'Include docstring (Purpose/Args/Returns/Example) and 5 rows of sample data.')
print('GENERATED CODE:'); print(ask(code_p, temperature=0.2, max_tokens=700))
var = ('Sep 2024: Actual 42.3Cr, Plan 38.0Cr, +4.3Cr (+11.3%).\n'
       'Driver: Enterprise closed 3 deals early. SMB missed 0.8Cr — 2 deals slipped.')
narr = ask(f'FP&A analyst. Board-level variance commentary. Plain English.\n2 sentences: (1) net variance + specific driver (2) offsetting factor + magnitude.\nData: {var}', temperature=0.2)
print('\nVARIANCE COMMENTARY:'); print(narr)

## Session 19 — AI for Strategy

In [ ]:
co = ('Indian SME accounting SaaS. 45Cr ARR, +60% YoY. 12,000 SME customers.\n'
      'Products: cloud accounting, GST, payroll. 180 people. Series B 120Cr.\n'
      'Competitor: Tally (legacy desktop). Trend: GST complexity driving cloud demand.')
p = ('Senior strategy consultant. SWOT analysis — specific, evidence-based, no generic observations.\n\n'
     f'Company: {co}\n\n'
     'For each quadrant: 3 specific points.\n## STRENGTHS / ## WEAKNESSES / ## OPPORTUNITIES / ## THREATS\n\n'
     'After SWOT: 50-word recommendation using Strength to capture Opportunity while mitigating one Threat.')
show(ask(p, temperature=0.4, max_tokens=800), 'SWOT Analysis')

## Session 20 — AI for Customer Support

In [ ]:
complaint = ('Customer: Raghav Mehta (3-year premium subscriber).\n'
             'Account downgraded to free with no notice. Lost all project data. Client demo in 4 hours. Account: PRO-44821.')
resp_p = ('Senior customer success specialist. Apply AEERF:\n'
          'A=Acknowledge | E=Empathise | E=Explain | R=Resolve (action+timeline) | F=Follow-up\n\n'
          f'Complaint: {complaint}\n\n'
          'Constraints: First name once. Under 200 words. Urgent, accountable, warm.\n'
          'Do NOT use: I understand your frustration, Certainly!, As per')
response = ask(resp_p, temperature=0.4)
print('RESPONSE:'); print(response)
eval_p = f'Evaluate against AEERF. Score each 1-5 with reason. Flag forbidden phrases. PASS (avg>=4) or NEEDS REVISION.\n\n{response}'
print('\nQUALITY EVAL:'); show(ask(eval_p, temperature=0.1))

---
# MODULE 5 — Advanced Prompt Engineering
## Sessions 21–25

## Session 21 — Multi-Step AI Workflows

In [ ]:
topic = 'Impact of rising interest rates on Indian real estate investment demand'
research = ask(f'5 key factors for how rising rates affect Indian real estate demand. Name + 2-sentence explanation + direction. Numbered list only.', temperature=0.3)
print('STEP 1:'); print(research)
draft = ask(f'Financial journalist. Opening 2 paragraphs for HNI investor audience. Narrative — not a list.\n\nResearch: {research}', temperature=0.5)
print('\nSTEP 2:'); print(draft)
polished = ask(f'Editor. Improve: eliminate passive voice, replace jargon, strengthen opening.\nOutput: improved version then list 3 changes.\n\nDraft: {draft}', temperature=0.3)
print('\nSTEP 3:'); show(polished)

## Session 22 — Advanced Frameworks (Tree of Thought)

In [ ]:
problem = 'B2B SaaS (30Cr ARR) needs 100Cr in 24 months. Growth: 35% YoY. Team: 60. Runway: 40Cr.'
tot = ('Strategic advisor using Tree of Thought.\n\n'
       f'Problem: {problem}\n\n'
       'PHASE 1: Generate 3 strategies: A=Product-led | B=Sales-led enterprise | C=Geographic expansion\n'
       'For each: 2-sentence description + one critical assumption.\n\n'
       'PHASE 2: Score each — Feasibility 1-5 | Risk 1-5 (1=low)\n\n'
       'PHASE 3: Recommend ONE with 3 specific 90-day actions + the one assumption it cannot get wrong.')
show(ask(tot, temperature=0.5, max_tokens=900), 'Tree of Thought Analysis')

## Session 23 — AI Automation (Batch Processing)

In [ ]:
leads = [
    {'id':1,'company':'TechVista Pvt Ltd','size':'200','industry':'SaaS','pain':'Manual reporting 8 hrs/week'},
    {'id':2,'company':'Heritage Textiles','size':'800','industry':'Manufacturing','pain':'Everything on paper'},
    {'id':3,'company':'FastCart','size':'50','industry':'E-commerce','pain':'500 tickets/day'},
    {'id':4,'company':'MedSync Clinics','size':'30','industry':'Healthcare','pain':'Fully manual scheduling'},
    {'id':5,'company':'Nexus Finance','size':'150','industry':'NBFC','pain':'Loan review 3 days each'},
]
results = []
for lead in leads:
    p = (f'B2B SaaS sales qualifier. Evaluate for our AI automation platform.\n'
         f'Company:{lead["company"]}|Size:{lead["size"]}|Industry:{lead["industry"]}|Pain:{lead["pain"]}\n'
         f'JSON only: {{"id":{lead["id"]},"fit_score":"1-10","priority":"HIGH/MEDIUM/LOW","reason":"one sentence"}}')
    try: results.append(json.loads(ask(p, temperature=0.0)))
    except: results.append({'id':lead['id'],'fit_score':'ERR','priority':'ERR','reason':'parse error'})
    time.sleep(0.5)
print(f'{"ID":<4}{"Company":<22}{"Score":<7}{"Priority":<9}Reason')
print('-'*70)
for r in results:
    c=leads[r['id']-1]['company']
    print(f'{r["id"]:<4}{c:<22}{str(r["fit_score"]):<7}{r["priority"]:<9}{r["reason"]}')

## Session 24 — Professional Prompt Library

In [ ]:
library = {}
def add_prompt(pid, title, cat, tech, ver, prompt_text, perf, variables):
    library[pid] = {'id':pid,'title':title,'category':cat,'technique':tech,'version':ver,
                    'prompt':prompt_text,'performance':perf,'variables':variables,'created':str(datetime.date.today())}
    print(f'Added: [{pid}] {title} ({ver})')
def get_prompt(pid, **kwargs):
    if pid not in library: return f'Not found: {pid}'
    t = library[pid]['prompt']
    for k,v in kwargs.items(): t = t.replace(f'{{{k}}}',v)
    return t
def list_library():
    print(f'{"ID":<15}{"Title":<35}{"Version":<10}Category'); print('-'*70)
    for e in library.values(): print(f'{e["id"]:<15}{e["title"]:<35}{e["version"]:<10}{e["category"]}')
add_prompt('MKT-001','LinkedIn Post Generator','marketing','persona+negative','v2.1.0',
    'You are {ROLE}. Brand voice: {BRAND_VOICE}. Topic: {TOPIC}.\nWrite 200-250 word LinkedIn post.\nHook: specific insight. Body: 3 points. CTA: open question.\nDo NOT start with I, use excited to share or game-changing.',
    {'success_rate':'87%','tested':45},['ROLE','BRAND_VOICE','TOPIC'])
add_prompt('HR-001','Inclusive JD Generator','hr','craft+negative','v1.3.0',
    'Senior HRBP. Write inclusive JD for {ROLE}. Must-haves: {MUST_HAVE}. Nice-to-have: {NICE_TO_HAVE}.\nLead with impact. No unnecessary degree requirements.',
    {'success_rate':'91%','tested':30},['ROLE','MUST_HAVE','NICE_TO_HAVE'])
list_library()
filled = get_prompt('MKT-001',ROLE='VP Marketing',BRAND_VOICE='Direct, evidence-based',TOPIC='Why vanity metrics kill marketing strategy')
show(ask(filled, temperature=0.6))

## Session 25 — Responsible AI

In [ ]:
jd = ('Seeking a rockstar Data Scientist — aggressive, dynamic, digital native with 8+ years.\n'
      'Thrives in fast-paced startup. Premier institution degree required.')
audit = ('Responsible AI auditor. Review JD for bias:\n'
         '1. Masculine-coded language (list word + neutral replacement)\n'
         '2. Ageist language (flag + replace)\n'
         '3. Elitist requirements (flag + remove)\n'
         '4. Exclusionary language\n\n'
         'Output: Issues table + revised JD that passes all checks.\n\n'
         f'JD: {jd}')
show(ask(audit, temperature=0.1), 'Bias Audit')

In [ ]:
safe = ('Employee in Finance consistently meets delivery targets but struggles with cross-functional communication.\n'
        'Draft one specific performance improvement goal for this area.')
print('SAFE INPUT (anonymised — correct approach):')
print(ask(safe, temperature=0.4))
print()
print('DATA PRIVACY — NEVER paste into consumer AI tools:')
for rule in ['Real names + performance issues','Salary/compensation data','Medical or leave history',
             'Customer PII (name, email, phone)','Undisclosed financial results','Legal case details']:
    print(f'  x {rule}')
print('\nAlways anonymise before using AI for HR, legal, or finance tasks.')

---
# MODULE 6 — Capstone Project & Certification
## Sessions 26–30

## Session 26 — Capstone Kickoff: Problem Definition

In [ ]:
# Customise these with YOUR actual project details
my_role    = 'Marketing Manager at a 300-person B2B software company'
my_problem = 'Weekly campaign commentary takes 3 hrs and produces inconsistent analysis across 4 team members'
frequency  = 'Weekly — every Monday morning'
cost       = '3 hrs x 4 analysts x 900/hr = 10,800/week'
p = ('Capstone project advisor for Prompt Engineering certification.\n'
     'Help me draft my Problem Definition Document.\n\n'
     f'Role: {my_role}\nProblem: {my_problem}\nFrequency: {frequency}\nCost: {cost}\n\n'
     'Output:\n1. PROBLEM STATEMENT (2 sentences — specific and measurable)\n'
     '2. PROPOSED AI SOLUTION TYPE\n3. SUCCESS CRITERIA (3 measurable metrics)\n'
     '4. SCOPE BOUNDARY (what this will NOT cover)\n5. CAPSTONE TRACK (A/B/C/D) with reason')
show(ask(p, temperature=0.4), 'Problem Definition Draft')

## Session 27 — Capstone Build Phase 1: Core Prompts

In [ ]:
# Example Capstone: Campaign Performance Commentary System
p1 = ('ROLE: Senior marketing analyst writing for non-marketing executives.\n'
      'CONTEXT: Weekly campaign summary. Audience: VP Sales and CFO.\n'
      'TASK: Interpret this week campaign data and write the performance narrative.\n'
      'INPUT: {CAMPAIGN_DATA}\n'
      'FORMAT: 3 paragraphs: headline vs target | top performer + why | underperformer + action.\n'
      'CONSTRAINTS: Under 200 words. Name specific campaigns. Numbers always in context.')
p2 = ('ROLE: Marketing FP&A specialist.\n'
      'CONTEXT: Board-level variance commentary.\n'
      'TASK: Write variance commentary for each campaign missing or beating target by >10%.\n'
      'INPUT: {VARIANCE_DATA}\n'
      'FORMAT: 2-sentence comment per campaign: result vs target + driver | action or watch item.\n'
      'CONSTRAINTS: Never say market conditions. Name the specific reason. Flag unknowns with [INVESTIGATE].')
p3 = ('ROLE: Growth marketing strategist.\n'
      'CONTEXT: End-of-week review. Leadership wants 3 next-week actions.\n'
      'TASK: Generate 3 specific next-week actions from performance data.\n'
      'INPUT: {PERFORMANCE_NARRATIVE} and {VARIANCE_COMMENTARY}\n'
      'FORMAT: 3 numbered items. Each: Action | Owner | Expected impact | Measurement.\n'
      'CONSTRAINTS: Actions achievable in 7 days. No generic advice.')
for i, p in enumerate([p1, p2, p3], 1): print(f'PROMPT {i}:\n{p}\n')
# Live test Prompt 1
sample = ('Campaign: Google Search | Spend:1.2L | Leads:48 | Target:50 | CPL:2500\n'
          'Campaign: LinkedIn Ads   | Spend:0.8L | Leads:12 | Target:20 | CPL:6667\n'
          'Campaign: Email Nurture  | Spend:0.1L | Leads:35 | Target:30 | CPL:286')
show(ask(p1.replace('{CAMPAIGN_DATA}', sample), temperature=0.3), 'Prompt 1 Live Test')

## Session 28 — Capstone QA: Quality Gates & Before/After Measurement

In [ ]:
weak_out = ('This week campaigns showed mixed results. Google Search was near target. '
            'LinkedIn underperformed. Email did well. We should optimise LinkedIn.')
rubric = ('Quality reviewer for marketing AI outputs.\n'
          'Score each 1-5 with specific reason:\n'
          '| Specificity (names campaigns/numbers) | Business language (no jargon) | Actionability | Accuracy | Length (<200 words) |\n'
          'Overall: PASS (avg>=4.0) or FAIL. If FAIL: provide rewritten version.\n\n'
          f'Output to evaluate: {weak_out}')
show(ask(rubric, temperature=0.1), 'Quality Gate Evaluation')
print('\nBEFORE/AFTER MEASUREMENT TEMPLATE:')
print(f'{"Dimension":<28} {"Before":<25} {"After Target"}')
print('-'*75)
for row in [('Time per report','3 hours (4 analysts)','<60 min (1 analyst)'),
            ('Format consistency','4 writers=4 formats','100% consistent'),
            ('Quality score','Baseline TBD','>=4.0/5 on rubric'),
            ('On-time rate','TBD (measure now)','>=95%')]:
    print(f'{row[0]:<28} {row[1]:<25} {row[2]}')

## Session 29 — Capstone Presentation Preparation

In [ ]:
project = ('AI-assisted weekly marketing commentary system.\n'
           '3-prompt pipeline: interpret, variance, recommend.\n'
           'Before: 3 hrs/week. After: 22 min/week. Quality: 4.2/5. Human approval mandatory.')
# Generate adversarial Q&A
qa_p = ('You are a tough, sceptical evaluator for a Prompt Engineering Capstone.\n'
        f'Project: {project}\n\n'
        'Generate the 6 hardest questions a sceptical evaluator could ask about THIS specific project.\n'
        'For each: provide a structured honest model answer (3-4 sentences).\n'
        'Focus on: bias risk, data accuracy, scalability, human dependency, ROI validity, responsible AI gaps.')
show(ask(qa_p, temperature=0.5, max_tokens=900), 'Adversarial Q&A Prep')

## Session 30 — Final Presentation & Certification

**Capstone Submission Checklist + Certification Standards**

In [ ]:
project = ('Marketing commentary system. 3-prompt pipeline. Before: 3hrs/week. After: 22min/week.\n'
           'Quality 4.2/5. Human approval required before sharing outputs.')
# Opening hook
hook_p = ('Professional presentation coach. Write the opening 90 seconds of a Capstone presentation.\n'
          f'Project: {project}\n\n'
          'Include: (1) Specific relatable problem (2) Quantified scale/cost (3) Implied tension (4) Bridge to solution.\n'
          'Format: Spoken script. Conversational but professional. Under 150 words.')
show(ask(hook_p, temperature=0.5), 'Opening Hook Script')
print('\nCAPSTONE SUBMISSION CHECKLIST:')
items = ['Problem Definition Document (specific, measured baseline)',
         'All prompts with metadata (ID, version, technique, performance notes)',
         'Demonstration outputs — before vs. after comparison',
         'Quantified Before/After (at least one measured metric)',
         'Responsible AI assessment (all 4 pillars, documented mitigations)',
         'Reflection Essay (failure analysis + genuine learning + what I would do differently)',
         '10-slide presentation deck',
         'Post-Certification 90-day development plan']
for i, item in enumerate(items,1): print(f'  [ ] {i}. {item}')
print('\nCERTIFICATION STANDARD: >=70% on written assessments + PASS on Capstone rubric')
print('Certification awarded for BOTH knowledge AND applied capability.')